# Pattern #2: Tool Use - Bridge to the Outside World

**From-Scratch Implementation**

## Overview

Tools enable agents to interact with the external world. In our system:
- **Internal Memory**: RAG retrieval (agent cognition)
- **External Action**: web_search (agent tool)

This notebook demonstrates using `web_search` (Tavily) for real-world operational information.

In [3]:
import sys
sys.path.append('..')

from utils import create_llm_provider, get_config
from tools import get_web_search_tool

# Initialize
config = get_config()
llm = create_llm_provider()
web_search = get_web_search_tool()

print(f"Using model: {config.get('model')}")
print(f"Web search provider: Tavily")
print(f"Timezone: {config.get('output.timezone')}")


Using model: openai/gpt-4o-mini
Web search provider: Tavily
Timezone: Asia/Colombo


### Validation (config from config/, web_search tool)

In [4]:
assert config.get("model"), "config.get('model') should be set"
assert config.get("output.timezone"), "output.timezone should be set"
assert llm is not None and callable(getattr(web_search, "search", None)), "LLM and web_search ready"
print("✓ Setup valid: config, LLM, and web_search ready.")

✓ Setup valid: config, LLM, and web_search ready.


## Baseline vs Tool-Enhanced Responses

Let's compare responses with and without web_search for operational queries.


In [5]:
def baseline_response(query: str) -> dict:
    """Generate response without web search tool."""
    system_prompt = """
You are a healthcare assistant for Sri Lankan hospitals.
Provide helpful information about hospital services.
Always end with: "This is educational information; verify with the hospital / consult your clinician."
""".strip()
    
    response = llm.generate(prompt=query, system_prompt=system_prompt)
    
    return {
        "response": response["response"],
        "tokens": response["total_tokens"],
        "latency_ms": response["latency_ms"],
        "sources": []
    }


def tool_enhanced_response(query: str) -> dict:
    """Generate response using web search tool."""
    import time
    
    start_time = time.time()
    
    # Step 1: Perform web search
    search_result = web_search.search(query)
    search_formatted = web_search.format_results(search_result)
    
    # Step 2: Generate response with search context
    system_prompt = """
You are a healthcare assistant with access to current web information.
Use the web search results to provide accurate, up-to-date operational information.
Always cite sources and include the checked timestamp.
Always end with: "This is educational information; verify with the hospital / consult your clinician."
""".strip()
    
    prompt = f"""
User Question: {query}

Web Search Results:
{search_formatted}

Please provide a comprehensive answer using the search results. Include URLs for verification.
""".strip()
    
    response = llm.generate(prompt=prompt, system_prompt=system_prompt)
    
    end_time = time.time()
    total_latency = int((end_time - start_time) * 1000)
    
    return {
        "response": response["response"],
        "tokens": response["total_tokens"],
        "search_tokens": 0,  # Tavily doesn't charge tokens
        "latency_ms": total_latency,
        "search_latency_ms": search_result["latency_ms"],
        "sources": [r["url"] for r in search_result.get("results", [])]
    }


def compare_approaches(query: str):
    """Compare baseline vs tool-enhanced."""
    print("=" * 80)
    print("BASELINE (No Tools)")
    print("=" * 80)
    
    baseline = baseline_response(query)
    print(f"\n{baseline['response']}\n")
    print(f"[Tokens: {baseline['tokens']}, Latency: {baseline['latency_ms']}ms]")
    print(f"Sources: None\n")
    
    print("=" * 80)
    print("TOOL-ENHANCED (With web_search)")
    print("=" * 80)
    
    enhanced = tool_enhanced_response(query)
    print(f"\n{enhanced['response']}\n")
    print(f"[Tokens: {enhanced['tokens']}, Latency: {enhanced['latency_ms']}ms]")
    print(f"Search Latency: {enhanced['search_latency_ms']}ms")
    print(f"Sources ({len(enhanced['sources'])}):")
    for url in enhanced['sources'][:3]:
        print(f"  - {url}")
    
    print("\n" + "=" * 80)
    print("COMPARISON")
    print("=" * 80)
    print(f"Latency increase: {enhanced['latency_ms'] - baseline['latency_ms']}ms")
    print(f"Token increase: {enhanced['tokens'] - baseline['tokens']}")
    print(f"Source verification: {'Yes (URLs provided)' if enhanced['sources'] else 'No'}")
    
    return {"baseline": baseline, "enhanced": enhanced}


## Example 1: Hospital Hours Query


In [6]:
query1 = "What are the OPD hours for Nawaloka Hospital Colombo?"
result1 = compare_approaches(query1)


BASELINE (No Tools)

Nawaloka Hospital in Colombo typically operates its Outpatient Department (OPD) from Monday to Saturday, with hours generally starting around 8:00 AM and closing by 5:00 PM. However, specific timings may vary based on the department or the day of the week. It's always best to check directly with the hospital for the most accurate and updated information regarding OPD hours.

This is educational information; verify with the hospital / consult your clinician.

[Tokens: 152, Latency: 4003ms]
Sources: None

TOOL-ENHANCED (With web_search)

Nawaloka Hospital in Colombo has its Outpatient Department (OPD) open 24 hours a day, seven days a week. This ensures that patients can access medical care at any time. Additionally, the hospital offers a drive-thru service that operates from 7 AM to 7 PM for convenience in accessing various healthcare services.

For more detailed information or specific inquiries, you can contact the hospital directly at 011 5577111.

For further ve

## Example 2: Emergency Contact


In [7]:
query2 = "What is the emergency hotline for Asiri Hospital?"
result2 = compare_approaches(query2)


BASELINE (No Tools)

The emergency hotline for Asiri Hospital in Sri Lanka is typically 011 2 300 300. However, it's always a good idea to verify this information directly with the hospital or check their official website for the most current contact details. 

This is educational information; verify with the hospital / consult your clinician.

[Tokens: 120, Latency: 1557ms]
Sources: None

TOOL-ENHANCED (With web_search)

The emergency hotline for Asiri Hospital is **1313**, and it operates 24/7 for urgent medical assistance. This number is available for anyone needing immediate help or guidance in emergency situations.

For more information, you can visit the official Asiri Health website: 
- [Asiri Hospital Emergency Services](https://asirihealth.com/services/accident-and-emergency)

This is educational information; verify with the hospital / consult your clinician.

[Tokens: 746, Latency: 4975ms]
Search Latency: 2821ms
Sources (5):
  - https://asirihealth.com/ongoing-number
  - http

## Pattern Summary

Tools extend agent capabilities to interact with the external world.

**Key Insights:**
- **Baseline**: LLM generates from training data (may be outdated/generic)
- **Tool-Enhanced**: Real-time, verified information with sources
- **web_search is the ONLY tool**: RAG is internal memory, not a tool

**When to use web_search:**
- Operational information (hours, contacts, addresses)
- Current announcements or changes
- Verification of time-sensitive facts
- Location-specific details

**Architecture:**
```
Internal Cognition: RAG retrieval (memory)
External Action: web_search (tool)
```

This separation is crucial for understanding agentic systems.
